# Lesson 09: Inference Optimization — Making AI Faster and Cheaper

## Learning Objectives
- Understand the performance metrics of AI inference: latency, throughput, cost
- Use code to measure and compare response speed across models
- Calculate AI usage costs and understand cost-reduction strategies
- Understand model compression techniques like quantization and distillation

> Fast, good, cheap — you can only pick two. The impossible triangle of AI Engineering.

## Environment Setup

> Please run `00_Environment_Setup.ipynb` first to set up dependencies and API keys, 
> then return to this notebook.

Once done, run the cell below to load environment variables:

In [ ]:
# Load API key from .env file (no need to enter it every time)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== Pick a provider: change this one line, nothing else =====
#   'openai'     cloud  needs OPENAI_API_KEY      strongest, has embeddings
#   'deepseek'   cloud  needs DEEPSEEK_API_KEY    cheapest cloud, no embeddings
#   'openrouter' cloud  needs OPENROUTER_API_KEY  many vendors, no embeddings
#   'ollama'     local  no key, free and offline  run `ollama serve` and pull the model first
PROVIDER = 'openai'

# All four speak the OpenAI API format. They differ only in URL, key, model names.
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = OpenAI's default endpoint
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # small model: cheap and fast
        'model_big': 'gpt-5.6-terra',                # big model: pricier and stronger
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # fast and cheap
        'model_big': 'deepseek-v4-pro',              # stronger and slower; both V4 models think first
        'embedding_model': None,                     # DeepSeek has no embeddings endpoint
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter does not proxy embeddings
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # local models ignore the key
        'model': 'gemma4:e2b-mlx',                   # run: ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # run: ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# Check the key first: with no key, the OpenAI client raises a long traceback.
if not cfg['api_key']:
    raise SystemExit(
        f"No API key for '{PROVIDER}'. Either add {PROVIDER.upper()}_API_KEY to your .env file,\n"
        f"or set PROVIDER = 'ollama' above to run locally with no key at all."
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# Every lesson below uses only these three names, so switching provider needs no
# other code change.
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'Connected! provider = {PROVIDER}, default model = {MODEL}')


---

## Activity 1: Measure Response Speed Across Models

### Activity Goal
Use code to precisely measure the response time (latency) of different models, and understand that larger models are not always slower — and that speed and quality involve trade-offs.

In [ ]:
import time
# Activity 1: Response speed comparison

prompt = 'Explain the concept of quantum computing in about 100 words.'
num_runs = 3  # Run each model 3 times and average

# Models to test
models_to_test = list(dict.fromkeys([MODEL, MODEL_BIG]))  # models you cannot access are skipped automatically

results = {}

for model_name in models_to_test:
    times = []
    token_counts = []
    print(f'\nTesting model: {model_name}')
    for i in range(num_runs):
        start = time.time()
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{'role':'user','content':prompt}],
                temperature=0.5,
                max_tokens=200)
        except Exception as e:
            print(f'  [SKIP] {model_name} unavailable: {str(e)[:100]}')
            break
        elapsed = max(time.time() - start, 1e-6)
        tokens = r.usage.completion_tokens
        times.append(elapsed)
        token_counts.append(tokens)
        print(f'  Run {i+1}: {elapsed:.2f}s, {tokens} tokens, speed: {tokens/elapsed:.0f} tok/s')

    if not times:  # model never ran, keep it out of the summary table
        continue
    avg_time = sum(times) / len(times)
    avg_tokens = sum(token_counts) / len(token_counts)
    results[model_name] = {
        'avg_time': avg_time,
        'avg_tokens': avg_tokens,
        'avg_speed': avg_tokens / avg_time
    }

print('\n' + '='*60)
print('Summary Comparison:')
print(f'{"Model":<20}{"Avg Time (s)":<15}{"Avg Tokens":<15}{"Speed (tok/s)":<15}')
print('-'*60)
for model_name, stats in results.items():
    print(f'{model_name:<20}{stats["avg_time"]:<15.2f}{stats["avg_tokens"]:<15.0f}{stats["avg_speed"]:<15.0f}')

print('\nPerformance metrics explained:')
print('  Time = from request sent to full response received')
print('  Tokens = how much text the AI generated (in tokens)')
print('  Speed = Tokens / Time (higher is faster)')

### Discussion
- Which model is fastest? Does that match your intuition?
- If an AI is 50% faster but 5% less accurate, would you accept the trade-off? Does it depend on the scenario?
- TTFT (Time To First Token) vs full response time — which affects user experience more?

---

## Activity 2: Calculate AI Usage Costs

### Activity Goal
Use real pricing information to estimate costs for different usage scenarios. This exercise builds awareness that "AI also costs money."

> Reference pricing (Aug 2026): gpt-5.6-luna input $0.20/1M tokens, output $1.20/1M tokens

In [ ]:
# Activity 2: AI usage cost estimation

# Pricing reference (USD per million tokens)
# Prices change — check each vendor's pricing page before class. These are
# order-of-magnitude references only.
pricing = {
    'gpt-5.6-luna': {'input': 0.20, 'output': 1.20},
    'gpt-5.6-terra': {'input': 2.00, 'output': 12.00},
    'deepseek-v4-flash': {'input': 0.14, 'output': 0.28},   # api-docs.deepseek.com, cache-miss input
    'deepseek-v4-pro': {'input': 0.435, 'output': 0.87},    # a cache hit makes input ~50x cheaper
}

# Scenario definitions
scenarios = [
    {'name': 'Personal daily use', 'users_per_day': 1, 'queries_per_user': 10,
     'input_tokens_per_query': 100, 'output_tokens_per_query': 200},
    {'name': 'Small startup', 'users_per_day': 500, 'queries_per_user': 5,
     'input_tokens_per_query': 300, 'output_tokens_per_query': 500},
    {'name': 'Enterprise support', 'users_per_day': 10000, 'queries_per_user': 3,
     'input_tokens_per_query': 200, 'output_tokens_per_query': 300},
]

print('AI Usage Cost Estimation (Monthly, USD)\n')
print(f'{"Scenario":<20}{"Model":<20}{"Daily Queries":<14}{"Monthly Cost":<14}{"Yearly Cost":<14}')
print('-'*82)

for scenario in scenarios:
    daily_queries = scenario['users_per_day'] * scenario['queries_per_user']
    for model_name, price in pricing.items():
        daily_input = daily_queries * scenario['input_tokens_per_query'] / 1_000_000
        daily_output = daily_queries * scenario['output_tokens_per_query'] / 1_000_000
        daily_cost = daily_input * price['input'] + daily_output * price['output']
        monthly_cost = daily_cost * 30
        yearly_cost = monthly_cost * 12
        print(f'{scenario["name"]:<20}{model_name:<20}{daily_queries:<14}${monthly_cost:<13.2f}${yearly_cost:<13.2f}')

print('\nKey Findings:')
print('1. gpt-5.6-luna is roughly 10x cheaper than gpt-5.6-terra')
print('2. For simple tasks, using a cheaper model can save significant cost')
print('3. Enterprise support can cost thousands per month — model choice matters!')

### Discussion
- Which number surprised you most?
- In your own scenario, how would you balance model choice and cost?
- When is it worth paying more for a better model?

---

## Activity 3: Understanding Model "Compression" — Intro to Quantization

### Activity Goal
Quantization is a key technique for reducing model cost and latency.
Intuitive analogy: compress a high-resolution image (32-bit weights) into a standard-resolution image (8-bit or 4-bit weights) — file size shrinks, but you can still recognize what it is.

In [ ]:
# Activity 3: Quantization concept demo

# Use numbers to simulate quantization effect
import random

# Simulate a model's weight matrix (32-bit floating-point)
print('Simulating model quantization effect:\n')

# Generate some simulated weights
random.seed(42)
weights_32bit = [random.uniform(-1, 1) for _ in range(10)]

# Simulate quantization to 4-bit (only 16 possible values)
def quantize_4bit(value):
    """Quantize a 32-bit float to 4-bit integer (0-15) and map back to original range"""
    # Map to 0-1 range
    normalized = (value + 1) / 2
    # Quantize to 0-15 (4-bit: 16 levels)
    quantized = round(normalized * 15)
    # Map back to -1 to 1
    restored = (quantized / 15) * 2 - 1
    return restored

weights_4bit = [quantize_4bit(w) for w in weights_32bit]

print('Original 32-bit weights vs Quantized 4-bit weights:')
print(f'{"Original (32-bit)":<20}{"Quantized (4-bit)":<20}{"Error":<10}')
print('-'*50)
total_error = 0
for w32, w4 in zip(weights_32bit, weights_4bit):
    error = abs(w32 - w4)
    total_error += error
    print(f'{w32:<20.6f}{w4:<20.6f}{error:<10.6f}')

print(f'\nAverage error: {total_error/10:.6f}')
print(f'Storage savings: {(1-4/32)*100:.0f}%')

print('\nPractical implications:')
print('  32-bit model = 70GB → 4-bit quantized model ≈ 9GB')
print('  After quantization, the model can run on an ordinary laptop!')
print('  While there is some precision loss, most tasks barely notice the difference.')

### Discussion
- If you could run a "good enough" AI model locally on your phone, what would you use it for?
- Quantization trades precision for efficiency — in what scenarios is that trade-off unacceptable?
- Why is a quantized model faster? (Hint: smaller data → faster memory reads)

---

## Lesson Review

| Skill | Description |
|-------|-------------|
| Speed measurement | Use code to precisely measure model response time and throughput |
| Cost estimation | Calculate AI usage costs for various scenarios using real pricing |
| Quantization concepts | Understand how quantization compresses models through simulation |

### Homework
1. Use the same approach to measure the speed of other AI services you use
2. Estimate your own yearly AI usage cost (if billed by API pricing)
3. Search for "GGUF quantized models" to learn about models you can run locally